# Coculture Untreated - Mechanical Automatic Modeling
Decode processed data, fit coculture untreated model zoo, and run sensitivity/uncertainty analysis.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

  Activating project at `~/Desktop/Research/CancerGrowthDynamics/Modeling_Approaches/02_mechanical_automatic_package`


In [2]:
using CSV, DataFrames, Plots, Dates
include(joinpath(@__DIR__, "..", "src", "MechanicalAutomaticModeling.jl"))
using .MechanicalAutomaticModeling
using GrowthParameterEstimation

In [3]:
condition = "coculture_untreated"
decoded = MechanicalAutomaticModeling.IOUtils.decode_condition_dataframe(condition; start=@__DIR__)
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
decoded_path = joinpath(out.csv, "$(condition)_automatic_decoded.csv")
CSV.write(decoded_path, decoded)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="decode", outputs=[decoded_path], start=@__DIR__)
first(decoded, min(10, nrow(decoded)))

Row,time,count,source_file,condition,density,cell_line,dose,mix
,Float64,Float64,String,String,String,String,Float64,String
1,1.0,15.17,measure_25-75A2780CoUntreat_2500Thres_well_day_averages.csv,coculture_untreated,,,0.0,
2,1.0,13.83,measure_25-75A2780CoUntreat_2500Thres_well_day_averages.csv,coculture_untreated,,,0.0,
3,1.0,9.42,measure_25-75A2780CoUntreat_2500Thres_well_day_averages.csv,coculture_untreated,,,0.0,
4,1.0,21.36,measure_25-75A2780CoUntreat_2500Thres_well_day_averages.csv,coculture_untreated,,,0.0,
5,1.0,11.12,measure_25-75A2780CoUntreat_2500Thres_well_day_averages.csv,coculture_untreated,,,0.0,
6,1.0,6.79,measure_25-75A2780CoUntreat_2500Thres_well_day_averages.csv,coculture_untreated,,,0.0,
7,1.0,37.23,measure_25-75A2780cisCoUntreat_4500Thres_well_day_averages.csv,coculture_untreated,,,0.0,
8,1.0,35.57,measure_25-75A2780cisCoUntreat_4500Thres_well_day_averages.csv,coculture_untreated,,,0.0,
9,1.0,21.05,measure_25-75A2780cisCoUntreat_4500Thres_well_day_averages.csv,coculture_untreated,,,0.0,


In [4]:
fit_artifacts = MechanicalAutomaticModeling.FitWorkflows.run_condition_fit!(decoded, condition; start=@__DIR__)
first(fit_artifacts.ranking, min(10, nrow(fit_artifacts.ranking)))

Row,model,sse,weighted_sse,aic,bic,n_params,delta_bic
,String,Float64,Float64,Float64,Float64,Int64,Float64
1,null_coculture,1.0e12,1.0e12,10757.0,10773.9,4,0.0
2,lotka_volterra_competition,1.0e12,1.0e12,10761.0,10786.3,6,12.4372
3,lotka_volterra_hill_competition,1.0e12,1.0e12,10771.0,10817.4,11,43.5302


In [5]:
analysis_artifacts = MechanicalAutomaticModeling.AnalysisWorkflows.run_condition_analysis!(decoded, fit_artifacts, condition; start=@__DIR__)
analysis_artifacts.sensitivity

Row,model,note
,String,String
1,null_coculture,Sensitivity call not available with current signature; update wrapper.


In [6]:
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
summary = DataFrame(
    condition = [condition],
    decoded_rows = [nrow(decoded)],
    fit_rows = [nrow(fit_artifacts.ranking)],
    sensitivity_rows = [nrow(analysis_artifacts.sensitivity)],
    generated_at = [Dates.format(now(UTC), dateformat"yyyy-mm-ddTHH:MM:SS")]
)
summary_path = joinpath(out.metrics, "$(condition)_automatic_summary.csv")
CSV.write(summary_path, summary)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="summary", outputs=[summary_path], start=@__DIR__)
summary

Row,condition,decoded_rows,fit_rows,sensitivity_rows,generated_at
,String,Int64,Int64,Int64,String
1,coculture_untreated,502,3,1,2026-04-23T23:21:45
